In [5]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>MPI Point-to-Point Communication: Blocking</font></center></h1>

# <font color='blue'>MPI Point-to-Point Communication: Blocking</font>

## Blocking communication

- <font color='green'>Definition:</font> a blocking communication does not return until the message data and envelope have been safely stored away so that the sender is free to modify the send buffer after return.
- The term blocking may be confusing. Indeed based on the definition above, one can infer:
  - The call to a send procedure does not obstruct the flow of the program at that line of the code up to the completion of the communication.  Therefore, a blocking sender may return when the transmission of the message may be:
    - not yet started
    - ongoing
    - completed (less likely)

## Point-to-Point Communication

<div style="text-align: center; font-size: 24px; max-width: 2000px;">
    <span">
        It is a communication between <font color='green'>two processes</font> where a sender (source process) sends message to a receiver (destination process).
    </span>
</div>

- Procedure (C/C++ binding, Fortran binding, Fortran 2008 binding)
- Message data
  - Buffer (<font color='green'>address</font>)
  - Datatype (basic or derived?)
  - Count (number of elements, <font color='green'>not bytes</font>)
- Message envelope
  - Source
  - Destination
  - Tag

## Basic Datatypes (C/C++)

<table style="border: 1px solid black; text-align: center">
  <tr>
    <th style="text-align: center; background-color: powderblue">MPI datatype</th><th style="text-align: center; background-color: powderblue">C datatype</th>
  </tr>
    <tr><td style="text-align: left">MPI_INT              </td><td style="text-align: left"> int            </td></tr>
    <tr><td style="text-align: left">MPI_UNSIGNED         </td><td style="text-align: left"> unsigned int   </td></tr>
    <tr><td style="text-align: left">MPI_FLOAT            </td><td style="text-align: left"> float          </td></tr>
    <tr><td style="text-align: left">MPI_DOUBLE           </td><td style="text-align: left"> double         </td></tr>
    <tr><td style="text-align: left">MPI_C_COMPLEX        </td><td style="text-align: left"> float_Complex  </td></tr>
    <tr><td style="text-align: left">MPI_C_DOUBLE_COMPLEX </td><td style="text-align: left"> double_Complex </td></tr>
    <tr><td style="text-align: left">MPI_C_BOOL           </td><td style="text-align: left"> _Bool          </td></tr>
    <tr><td style="text-align: left">MPI_CHAR             </td><td style="text-align: left"> char           </td></tr>
    <tr><td style="text-align: left">MPI_BYTE             </td><td style="text-align: left"> -----          </td></tr>
    <tr><td style="text-align: left">MPI_PACKED           </td><td style="text-align: left"> -----          </td></tr>
    <tr><td style="text-align: center" colspan="2">and many more -> <a href="https://www.mpi-forum.org/docs">https://www.mpi-forum.org/docs</a> </td></tr>
</table>

## Basic Datatypes (Fortran)

<table style="border: 1px solid black; text-align: center">
  <tr>
    <th style="text-align: center; background-color: powderblue">MPI datatype</th><th style="text-align: center; background-color: powderblue">Fortran datatype</th>
  </tr>
    <tr><td style="text-align: left">MPI_INTEGER          </td><td style="text-align: left"> integer          </td></tr>
    <tr><td style="text-align: left">MPI_REAL             </td><td style="text-align: left"> real(kind=4)     </td></tr>
    <tr><td style="text-align: left">MPI_DOUBLE_PRECISION </td><td style="text-align: left"> real(kind=8)     </td></tr>
    <tr><td style="text-align: left">MPI_COMPLEX          </td><td style="text-align: left"> complex(kind=4)  </td></tr>
    <tr><td style="text-align: left">MPI_DOUBLE_COMPLEX   </td><td style="text-align: left"> complex(kind=8)  </td></tr>
    <tr><td style="text-align: left">MPI_LOGICAL          </td><td style="text-align: left"> logical          </td></tr>
    <tr><td style="text-align: left">MPI_CHARACTER        </td><td style="text-align: left"> character(len=1) </td></tr>
    <tr><td style="text-align: left">MPI_BYTE             </td><td style="text-align: left"> -----            </td></tr>
    <tr><td style="text-align: left">MPI_PACKED           </td><td style="text-align: left"> -----            </td></tr>
    <tr><td style="text-align: center" colspan="2">and many more -> <a href="https://www.mpi-forum.org/docs">https://www.mpi-forum.org/docs</a> </td></tr>
</table>

## The Ping-Pong example

<!-- <img src="figs/pingpong.svg" alt="MPI-standard" align="right" width="35%"/><p> </p> -->
```{image} figs/pingpong.svg
:width: 300px
```
- Consider two processes with ranks 0 and 1
- Rank 0 sends a message to rank 1
- Rank 1 receives it and sends it back to rank 0
- The operation can be <font color='green'>repeated</font> several times.

::: {.callout-note}
<span style="color:darkblue">
Initial data exchanges in the Ping-Pong algorithm may incur significantly higher latency than subsequent ones due to MPI communication initialization. Consequently, one should be cautious when measuring runtime to avoid skewed results.</span>
:::


## The Ping-Pong example

- In this example, the ping-pong is done only once.
- The code in C ( or Fortran) will be shown to you. Then, think about the following questions:
  - Does rank 0 receive the initial value of rank 1, if not, how can it be done?
  - If we add a loop enclosing the data transfer lines of the program and repeat the operation, would the program run without any problem?
  - Does the program have a deadlock problem? If not, can the problem be introduced to the program only by rearranging the orders of send/receive?


<div style="text-align: center; font-size: 24px; max-width: 2000px;">
    <span">
        A <font color='green'>deadlock</font> is a scenario in which a process is trying to exchange data to another process but there is <font color='green'>no match</font>, e.g. it is ready to send a data but the other process is not and will not be prepared to accept or the opposite case, i.e. the process is waiting to receive but the other is not sending and will not send a <font color='green'>matching</font> message.
    </span>
</div>

## Single-round ping-pong in C

```cpp
#include <mpi.h>
#include <stdio.h>
int main(int argc, char **argv) {
int ierr, irank, nrank;
MPI_Status status;
double d=0.0;
ierr=MPI_Init(&argc,&argv);
ierr=MPI_Comm_rank(MPI_COMM_WORLD,&irank);
ierr=MPI_Comm_size(MPI_COMM_WORLD,&nrank);
if(irank==0) d=100.0;
if(irank==1) d=200.0;
printf("BEFORE: nrank,irank,d = %5d%5d%8.1f\n",nrank,irank,d);
if(irank==0) {
    MPI_Send(&d,1,MPI_DOUBLE,1,11,MPI_COMM_WORLD);
    MPI_Recv(&d,1,MPI_DOUBLE,1,22,MPI_COMM_WORLD,&status);
}
else if(irank==1) {
    MPI_Recv(&d,1,MPI_DOUBLE,0,11,MPI_COMM_WORLD,&status);
    MPI_Send(&d,1,MPI_DOUBLE,0,22,MPI_COMM_WORLD);
}
printf("AFTER: nrank,irank,d = %5d%5d%8.1f\n",nrank,irank,d);
ierr=MPI_Finalize();
}
```

## Single-round ping-pong in Fortran

```fortran
program pingpong
use mpi_f08
implicit none
integer:: irank, nrank, ierr
real(kind=8):: d=0.d0
type(MPI_STATUS):: status
call MPI_INIT(ierr)
call MPI_COMM_RANK(MPI_COMM_WORLD,irank,ierr)
call MPI_COMM_SIZE(MPI_COMM_WORLD,nrank,ierr)
if(irank==0) d=100.d0
if(irank==1) d=200.d0
write(*,'(a,2i5,f8.1)') 'BEFORE: nrank,irank,d = ',nrank,irank,d
if(irank==0) then
    call MPI_SEND(d,1,MPI_DOUBLE_PRECISION,1,11,MPI_COMM_WORLD,ierr)
    call MPI_RECV(d,1,MPI_DOUBLE_PRECISION,1,22,MPI_COMM_WORLD,status,ierr)
elseif(irank==1) then
    call MPI_RECV(d,1,MPI_DOUBLE_PRECISION,0,11,MPI_COMM_WORLD,status,ierr)
    call MPI_SEND(d,1,MPI_DOUBLE_PRECISION,0,22,MPI_COMM_WORLD,ierr)
endif
write(*,'(a,2i5,f8.1)') 'AFTER: nrank,irank,d = ',nrank,irank,d
call MPI_FINALIZE(ierr)
end program pingpong

```

## Inspection on question 3 of exercise 1


- Let’s consider two changes in the solution code of exercise 1:
  1. Both processes call first MPI_SEND and then MPI_RECV
  2. The buffer is not a scalar but an array whose length is determined at run time as an argument:

|                              |                                   |
|------------------------------|:----------------------------------|
|`mpirun –n 2 ./a.out 10`      |<font color='green'># OK</font><br>|
|`mpirun –n 2 ./a.out 100`     |<font color='green'># OK</font><br>|
|`mpirun –n 2 ./a.out 1000`    |<font color='green'># OK</font><br>|
|`mpirun –n 2 ./a.out 10000`   |<font color='green'># OK</font><br>|
|`mpirun –n 2 ./a.out ????????`|<font color='red'># at some array length DEADLOCK occurs</font> |

## Communication modes

There are four send communication modes:
         
<table style="border: 1px solid black; text-align: center">
  <tr>
    <th style="text-align: center; background-color:powderblue">Mode</th><th style="text-align: center; background-color:powderblue">Binding</th>
  </tr>
    <tr><td style="text-align: left">Synchronous            </td><td style="text-align: left">MPI_Ssend</td></tr>
    <tr><td style="text-align: left">Buffered (asynchronous)</td><td style="text-align: left">MPI_Bsend</td></tr>
    <tr><td style="text-align: left">Standard               </td><td style="text-align: left">MPI_Send </td></tr>
    <tr><td style="text-align: left">Ready                  </td><td style="text-align: left">MPI_Rsend</td></tr>
</table>

- <font color='green'>There is only one receive communication mode:</font>
  - Standard: MPI_Recv

## Synchronous send

- It can be started whether or not a matching receive was posted
- It will complete successfully only if a matching receive is posted
  - send buffer can be reused
  - receiver has reached a certain point in its execution
<!-- <img src="figs/Synchronous-send.svg" alt="MPI-layers" align="right" width="30%"/><p> </p> -->
```{image} figs/Synchronous-send.svg
:width: 800px
```
<font color='green'>Tips</font>
- Useful for debugging
- Serialization
- High latency (synchronization overhead)
- Best bandwidth

## Standard send

- It can be started whether or not a matching receive was posted
- It may complete before a matching receive is posted
  - Send buffer can be reused
  - The operation is local or nonlocal

<!-- <img src="figs/Standard-send.svg" alt="MPI-layers" align="right" width="30%"/><p> </p> -->
```{image} figs/Standard-send.svg
:width: 800px
```
<font color='green'>Tips</font>
- Deadlock may occur
- Minimal transfer time

::: {.callout-note}
<span style="color:darkblue">
The standard send is the standard choice for you!</span>
:::

## Example: Shift operation across a chain of processes

<!-- <img src="figs/ring-shift-closed-1.svg" alt="parallel execution" align="right" width="25%"/><p> </p> -->
```{image} figs/ring-shift-closed-1.svg
:width: 1000px
```

- Simplistic send/recv
  - <font color='red'>pairing is not reliable</font>

```cpp
//my left neighbor
left=(rank-1)%size;
//my right neighbor
right=(rank+1)%size;
MPI_Send(sendbuf,n,type,right,tag,comm);
MPI_Recv(recvbuf,n,type,left,tag,comm,status);
```

## Point-to-Point Communication MPI_Sendrecv

```cpp
int MPI_Sendrecv(const void *sendbuf, int sendcount, MPI_Datatype sendtype,int dest, int sendtag,
                       void *recvbuf, int recvcount, MPI_Datatype recvtype, int source, int recvtag,
                       MPI_Comm comm, MPI_Status *status)
```
- Syntax: simple combination of send and receive arguments
- MPI takes care, thereby <font color='green'>no deadlocks</font> occur
::: {.callout-note}
<span style="color:darkblue">
&#9642; disjoint send/receive buffers <br>
&#9642; can have different count & data type <br>
&#9642; blocking call</span>
:::
```cpp
// Rank left from myself
left = (rank – 1 + size) % size;
// Rank right from myself
right = (rank + 1) % size;
MPI_Sendrecv(sendbuf, n, MPI_INT, right, 0,
             recvbuf, n, MPI_INT, left,  0,
             MPI_COMM_WORLD, status);
```

<!-- <img src="figs/ring-shift-closed-2.svg" alt="parallel execution" align="right" width="25%"/><p> </p> -->
```{image} figs/ring-shift-closed-2.svg
:width: 1000px
```

## Point-to-Point Communication MPI_SENDRECV

- useful for open chains/non-circular shifts:
```cpp
// Rank left from myself.
left = rank – 1; if (left < 0) { left = MPI_PROC_NULL; }
// Rank right from myself.
right = rank + 1; if (right >= size) {right = MPI_PROC_NULL; }
MPI_Sendrecv(buffer_send, n, MPI_INT, right, 0,
             buffer_recv, n, MPI_INT, left,  0, MPI_COMM_WORLD, &status);
```
<!-- <img src="figs/ring-shift-open.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/ring-shift-open.svg
:width: 1000px
```
- <font color='green'>MPI_PROC_NULL</font> as source/destination acts as no-op
  - send/recv with MPI_PROC_NULL return immediately, buffers are not altered
- MPI_Sendrecv matches with simple *send/*recv point-to-point calls

## Serialization: Loss of efficiency

- Ring shift communication pattern: non-circular shifts
  - No concern over deadlock
  - Serialization
    - MPI_Send with rendezvous protocol
    - MPI_Ssend
<!-- <img src="figs/serialization.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/serialization.svg
:width: 1000px
```

## Pattern: ghost cell exchange

Many iterative algorithms require exchange of domain boundary layers

<!-- <img src="figs/ghost-cell.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/ghost-cell.svg
:width: 1000px
```

Many iterative algorithms require exchange of domain boundary layers


2D domain distributed to ranks (here 4 x 3), each rank gets one tile

<!-- <img src="figs/ghost-cell-4.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/ghost-cell-1.svg
:width: 300px
```
- Each rank’s tile is surrounded by <font color='green'>ghost cells</font>, representing the cells of the neighbors
  - ghost cell:
<!-- <img src="figs/ghost-cell-2.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/ghost-cell-2.svg
:width: 300px
```
After each sweep over a tile, perform <font color='green'>ghost cell exchange</font>, i.e., update ghost cells with new values of neighbor cells
<!-- <img src="figs/ghost-cell-3.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/ghost-cell-3.svg
:width: 300px
```

- Possible implementation:
  - copy new data into contiguous send buffer
  - send to corresponding neighbor receive new data from same neighbor
  - copy new data into ghost cells


```cpp
MPI_Sendrecv(
sb, …, j, 
rb, …, j, …)
```
<!-- <img src="figs/ghost-cell-4.svg" alt="world-communicator" align="right" width="40%"/><p> </p> -->
```{image} figs/ghost-cell-4.svg
:width: 300px
```
```cpp
MPI_Sendrecv(
sb, …, i, 
rb, …, i, …)
```

## Combined send and recv

- MPI_Sendrecv <font color='green'>combines a blocking send and receive</font> into a single API call. - Deadlocks are still possible if envelope does not match.
- Send and receive buffers <font color='green'>must not overlap</font>:
  - For specific cases: MPI_Sendrecv_replace

## MPI_Sendrecv

- C/C++ binding:
```cpp
int MPI_Sendrecv(const void *sendbuf, int sendcount, MPI_Datatype sendtype, int dest, int sendtag,
                       void *recvbuf, int recvcount, MPI_Datatype recvtype, int source, int recvtag,
                       MPI_Comm comm, MPI_Status *status)
```

- <font color='green'>sendbuf</font>: address of the first entry of the buffer to be sent
- <font color='green'>sendcount</font>: number of elements to be sent
- <font color='green'>sendtype</font>: type of the send data
- <font color='green'>recvbuf, recvcount, recvtype</font>: similarly for the receiving data
- <font color='green'>dest</font>: rank of the destination process within the communicator comm
- <font color='green'>sendtag</font> and <font color='green'>recvtag</font>: can have different values!

## Exercise 3:

1. Laplace equation in 1D: $\frac{d^2V}{dx^2}=0$ with Dirichlet BCs:
   $$V(x)|_{x=0}=0 \;\;\text{and}\;\; V(x)|_{x=1}=1$$
   - Analytical solution is available, namely $V(x)=x$
3. Discretization and using the Jacobi method leads to <font color='green'>stencil</font> algorithm: <br>
$$\color{green}V_i=\frac{V_{i+1}+V_{i-1}}{2}$$
4. Domain decomposition: each domain needs the boundary values
   - should be supplied by the neighboring processes of both sides
   - leads to a <font color='green'>double shift operation</font>




## Miscellaneous points

- Predefined macros:
  - <font color='green'>Wild cards</font>: `MPI_ANY_SOURCE`, `MPI_ANY_TAG`
  - Info about MPI standard: `MPI_VERSION`, `MPI_SUBVERSION`
  - Others: `MPI_SUCCESS`, `MPI_PROC_NULL`, ...
- <font color='green'>MPI_Status</font>: can be avoided `MPI_STATUS_IGNORE`
  - It is a structure in C/C++: `MPI_SOURCE`, `MPI_TAG`, `MPI_ERROR`, ...
  - In Fortran:
    - It used to be an integer array of size `MPI_STATUS_SIZE`.
    - Now a derived-type is available as well.
- Some other C/C++ and Fortran bindings: `MPI_Probe`, `MPI_Get_count`, ...

## Quiz:

1. How many different MPI point-to-point send modes exist?
   <ol style="list-style-type: lower-alpha;">
   <li>1</li>
   <li>2</li>
   <li>3</li>
   <li>4</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> d.
   </details>
2. If you send two messages $m_1$ and $m_2$ from rank $i$ to rank $j$ using PtP bindings, is it possible that the transfer of $m_2$ overtakes, i.e., be received before $m_1$? <br>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> No, messages in PtP communication were in the past subject to <font color='red'>nonovertaking</font> rule. In recent MPI standards, it is the default behavior but it can be overridden.
   </details>
3. What are the major risks of synchronous send? Is any of the same risks a concern also for the standard send? <br>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> (i) Deadlock, (ii) high latency. Both can be of concern for standard send.
   </details>
4. Does the receive process have to know the <font color='green'>rank</font> of the send process and the <font color='green'>tag</font> of the message? <br>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> No, it is <font color='red'>not necessary</font>. One can use wild cards such as <code> MPI_ANY_SOURCE</code> and <code>MPI_ANY_TAG</code>.
   </details>


